# Comparación: article (procesado) vs 1024seq

Este notebook carga los ficheros procesados del artículo y los `1024seq` existentes, extrae características simples (media, desviación estándar, energía, frecuencia pico vía PSD) y muestra tablas comparativas y tests estadísticos (KS).

In [7]:
import numpy as np
import pandas as pd
from scipy.signal import welch
from scipy.stats import ks_2samp

# Paths (workspace-relative)
paths = {
    'article_AF': 'PULSOVITAL/Metricas/sssd_article_AF_proc_proc_3000.npy',
    'article_NSR': 'PULSOVITAL/Metricas/sssd_article_NSR_proc_proc_3000.npy',
    '1024_AF': '1024seq_AF_normalized.npy',
    '1024_NSR': '1024seq_NSR.npy',
}
FS = 300.0  # frecuencia de muestreo usada en el pipeline objetivo
print('Will load:', paths)

Will load: {'article_AF': 'PULSOVITAL/Metricas/sssd_article_AF_proc_proc_3000.npy', 'article_NSR': 'PULSOVITAL/Metricas/sssd_article_NSR_proc_proc_3000.npy', '1024_AF': '1024seq_AF.npy', '1024_NSR': '1024seq_NSR.npy'}


In [8]:
def load_and_squeeze(p):
    a = np.load(p)
    # expected shapes: (1024,3000,1) or (N,L,1) or (N,L)
    if a.ndim == 3 and a.shape[2] == 1:
        a = a[:, :, 0]
    elif a.ndim == 1:
        a = a[np.newaxis, :]
    return a

import os, glob
datasets = {}
# helper: attempt several fallback locations/patterns when a given path is missing
def find_alternative(path, keyword=None):
    # if exact exists, return it
    if os.path.exists(path):
        return path
    # try in PULSOVITAL/Metricas with variations
    base = os.path.basename(path)
    candidates = []
    # direct basename in metrícas
    candidates += glob.glob(os.path.join('PULSOVITAL','Metricas','*'+base+'*'))
    # look for files containing keyword (e.g., 'AF' or 'NSR')
    if keyword is not None:
        candidates += glob.glob(os.path.join('PULSOVITAL','Metricas','*'+keyword+'*.npy'))
        candidates += glob.glob('*'+keyword+'*.npy')
    # prefer exact base match first
    for c in candidates:
        if os.path.basename(c) == base:
            return c
    # otherwise return first candidate if any
    return candidates[0] if candidates else path

for name, p in paths.items():
    # derive keyword from name (e.g., 'AF' or 'NSR' or '1024')
    keyword = None
    if 'AF' in name.upper():
        keyword='AF'
    elif 'NSR' in name.upper():
        keyword='NSR'
    alt = find_alternative(p, keyword=keyword)
    if alt != p and os.path.exists(alt):
        print(f'Using alternative for {name}: {alt}')
        p = alt
    if not os.path.exists(p):
        print(f'Missing file: {p}')
        datasets[name] = np.empty((0,), dtype=np.float32)
        continue
    try:
        datasets[name] = load_and_squeeze(p)
    except Exception as e:
        print(f'Failed loading {p}: {e}')
        datasets[name] = np.empty((0,), dtype=np.float32)
for k, v in datasets.items():
    print(k, getattr(v, 'shape', None), getattr(v, 'dtype', None))

Missing file: PULSOVITAL/Metricas/sssd_article_AF_proc_proc_3000.npy
Missing file: PULSOVITAL/Metricas/sssd_article_NSR_proc_proc_3000.npy
Missing file: 1024seq_AF.npy
Missing file: 1024seq_NSR.npy
article_AF (0,) float64
article_NSR (0,) float64
1024_AF (0,) float64
1024_NSR (0,) float64


In [9]:
def extract_features(signals, fs=FS, fmax=50):
    # signals: (N, L)
    rows = []
    for sig in signals:
        mu = float(np.mean(sig))
        sd = float(np.std(sig))
        energy = float(np.mean(sig.astype('float64')**2))
        # PSD peak frequency
        try:
            f, P = welch(sig, fs=fs, nperseg=min(1024, len(sig)))
            # limit to fmax
            mask = f <= fmax
            fpeak = float(f[mask][np.argmax(P[mask])]) if np.any(mask) else float(f[np.argmax(P)])
        except Exception:
            fpeak = float('nan')
        rows.append({'mean': mu, 'std': sd, 'energy': energy, 'peak_freq': fpeak})
    return pd.DataFrame(rows)

features = {}
for name, arr in datasets.items():
    # handle empty loaded arrays by creating an empty features DataFrame with expected columns
    if getattr(arr, 'size', 0) == 0:
        features[name] = pd.DataFrame(columns=['mean','std','energy','peak_freq'])
    else:
        features[name] = extract_features(arr)
# show first rows for each dataset
for name, df in features.items():
    print('\n===', name, '===')
    if df.empty:
        print('(empty dataset)')
    else:
        display(df.head())


=== article_AF ===
(empty dataset)

=== article_NSR ===
(empty dataset)

=== 1024_AF ===
(empty dataset)

=== 1024_NSR ===
(empty dataset)


In [10]:
# Aggregate summaries (dataset-level): mean of features, and counts
summary_rows = []
for name, df in features.items():
    if df.empty:
        # create placeholder summary with NaNs and count=0 for expected features
        summary = pd.DataFrame(index=['mean','std','energy','peak_freq'], columns=['count','mean','std','median'])
        summary[:] = np.nan
        summary['count'] = 0
    else:
        summary = df.agg(['count','mean','std','median']).transpose()
    summary['dataset'] = name
    summary_rows.append(summary.reset_index().rename(columns={'index':'feature'}))

# Concatenate into a single table for easy inspection
summary_table = pd.concat(summary_rows, ignore_index=True)
# pivot for readability: rows=dataset, cols=feature-stat (flattened)
pivot = summary_table.pivot_table(index='dataset', columns='feature')
# flatten MultiIndex columns
pivot.columns = [f"{stat}_{feat}" for stat, feat in pivot.columns]
pivot = pivot.reset_index()
display(pivot)

,dataset,count_energy,count_mean,count_peak_freq,count_std
0,1024_AF,0.0,0.0,0.0,0.0
1,1024_NSR,0.0,0.0,0.0,0.0
2,article_AF,0.0,0.0,0.0,0.0
3,article_NSR,0.0,0.0,0.0,0.0


In [11]:
# Pairwise KS-tests between matching classes (AF vs AF, NSR vs NSR) and overall feature comparisons
pairs = [ ('article_AF','1024_AF'), ('article_NSR','1024_NSR') ]
ks_results = []
for a,b in pairs:
    fa = features[a] ; fb = features[b]
    for col in fa.columns:
        stat, p = ks_2samp(fa[col].dropna(), fb[col].dropna())
        ks_results.append({'pair': f'{a} vs {b}', 'feature': col, 'ks_stat': float(stat), 'p_value': float(p)})
ks_df = pd.DataFrame(ks_results)
display(ks_df)

C:\Users\BISITE-NEL\AppData\Local\Temp\ipykernel_17112\664194064.py:7: SmallSampleWarning: One or more sample arguments is too small; all returned values will be NaN. See documentation for sample size requirements.
  stat, p = ks_2samp(fa[col].dropna(), fb[col].dropna())


,pair,feature,ks_stat,p_value
0,article_AF vs 1024_AF,mean,NaN,NaN
1,article_AF vs 1024_AF,std,NaN,NaN
2,article_AF vs 1024_AF,energy,NaN,NaN
3,article_AF vs 1024_AF,peak_freq,NaN,NaN
4,article_NSR vs 1024_NSR,mean,NaN,NaN
5,article_NSR vs 1024_NSR,std,NaN,NaN
6,article_NSR vs 1024_NSR,energy,NaN,NaN
7,article_NSR vs 1024_NSR,peak_freq,NaN,NaN


## Siguientes pasos
- Si quieres métricas adicionales (MMD, DTW, C-FID) puedo integrar funciones desde `notebooks/compare_synthetic_vs_real_report.ipynb`.
- También puedo guardar estas tablas en CSV/HTML para revisión fuera del notebook.

In [ ]:
# Mostrar tabla de métricas calculada
from IPython.display import HTML, display
p = 'notebooks/outputs/pair_metrics_variant.html'
if os.path.exists(p):
    display(HTML(open(p, 'r', encoding='utf-8').read()))
else:
    print('No se encontró', p)
